In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from tqdm import tqdm
from fastai.vision.all import load_learner
from mtrain.utils import mkdir, show, draw_grid
from mtrain.smallnet.unet.predict import overlay_mask_on_img
import torch
import itertools

In [ ]:
DS = Path("../../datasets")
BASE = DS / "test-samples"
NEG_MASKING_DIR = mkdir(BASE / "neg-masking")
NEG_MASK_WORK_DIR = mkdir(BASE / "neg-masking" / "V1")

In [ ]:
class DiskImage:
    @classmethod
    def save(cls, arr: np.ndarray, path: Path | str):
        Image.fromarray(arr, "RGB").save(path)

    @classmethod
    def load(cls, path: Path | str):
        return np.array(Image.open(path).convert("RGB"))


class DiskBooleanMask:
    @classmethod
    def save(cls, arr: np.ndarray, path: Path | str):
        Image.fromarray(arr, "L").save(path)

    @classmethod
    def load(cls, path: Path | str):
        return np.array(Image.open(path).convert("L"))

    @classmethod
    def load_as_bool(cls, path: Path | str):
        return np.array(Image.open(path).convert("L")).astype(bool)


class DiskMaskJson:
    @classmethod
    def save(cls, arr: np.ndarray, path: Path | str):
        with open(path, "w") as f:
            json.dump(arr.tolist(), f)

    @classmethod
    def load(cls, path: Path | str):
        with open(path) as f:
            return np.array(json.load(f))

# Generate input masks and images

In [ ]:
IMAGE_DIR = BASE / "positive-samples"
images = list(IMAGE_DIR.glob("*.jpg"))
len(images)

In [ ]:
# generate the masks first
learner100 = load_learner(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
)
SIZE = 100

In [ ]:
from mtrain.smallnet.unet.predict.strided import multiple, single


img = plt.imread("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/test-pos-samples/stridded_mapillary_neg_with_elev/scale=50_to_50/1102231600254227/image.jpg")
mask = single.predict_unet_only_mask(img, 100, learner100, 4)

In [ ]:
from mtrain.smallnet.unet.predict import overlay_mask_on_img
res = overlay_mask_on_img(img, mask.astype(bool))
show([res], ncols=1)

In [ ]:
from mtrain.smallnet.unet.predict.strided import multiple, single
from PIL import Image
import shutil
from tqdm import tqdm


def run_model_on_images(images, learner, sz, results_dir, bs):
    for im in tqdm(images):
        dest = mkdir(results_dir / im.stem)
        shutil.copy(im, dest / f"image{im.suffix}")
        img = plt.imread(im)
        mask_dest = dest / "mask.png"
        res_dest = dest / "res.jpg"
        if mask_dest.exists() and res_dest.exists():
            continue
        masks = multiple.strided_predict_unet_only_mask([img], sz, learner, [50], bs)
        mask = masks[0]
        overlaid = overlay_mask_on_img(img, mask)

        Image.fromarray(mask, "L").save(dest / "mask.png")
        Image.fromarray(overlaid, "RGB").save(dest / "res.jpg")

In [ ]:
from mtrain.smallnet.unet.predict.strided import multiple, single
from PIL import Image
import shutil
from tqdm import tqdm


def get_single_strided_results(images, learner, sz, bs):
    imgs = [plt.imread(im) for im in images]
    return [
        single.strided_predict_unet_only_mask(img, sz, learner, [50], bs)
        for img in tqdm(imgs)
    ]


def get_multiple_strided_results(images, learner, sz, bs):
    imgs = [plt.imread(im) for im in images]
    return multiple.strided_predict_unet_only_mask(imgs, sz, learner, [50], bs)


# def test_strided_mult(images, learner, sz, bs):
#     sng = get_single_strided_results(images, learner, sz, bs)
#     mult = get_multiple_strided_results(images, learner, sz, bs)

#     assert len(sng) == len(mult)
#     for s,m in zip(sng, mult):
#         assert np
# for im in tqdm(images):
#     dest = mkdir(results_dir / im.stem)
#     shutil.copy(im, dest / f"image{im.suffix}")
#     img = plt.imread(im)
#     mask_dest = dest / "mask.png"
#     res_dest = dest / "res.jpg"
#     if mask_dest.exists() and res_dest.exists():
#         continue
#     multiple_mask = multiple.strided_predict_unet_only_mask([img], sz, learner, [50], bs)[0]
#     single_mask = single.strided_predict_unet_only_mask(img, sz, learner, [50], bs)
#     assert np.all(multiple_mask == single_mask)
#     mask = multiple_mask
#     overlaid = overlay_mask_on_img(img, mask)

#     Image.fromarray(mask, "L").save(dest / "mask.png")
#     Image.fromarray(overlaid, "RGB").save(dest / "res.jpg")

In [ ]:
sng = get_single_strided_results(images[:10], learner100, 100, 4)
print("done single")
mult = get_multiple_strided_results(images[:10], learner100, 100, 8)

In [ ]:
len(sng), len(mult)

In [ ]:
for s, m in zip(sng, mult):
    assert np.all(s == m)

In [ ]:
MBS = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/iter_4_engulf_t009_more-skew-resnet18-50x50-v2"
)
learn50 = load_learner(MBS / "model.pkl")

In [ ]:
import shutil

IMAGE = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100/1446686046883573/image.jpg"
)
shutil.rmtree("./test-batch", ignore_errors=True)
mask, overlaid = next(
    run_model_on_images([IMAGE], learner100, 100, Path("./test-batch"), 8)
)

In [ ]:
test_strided_mult([IMAGE], learner100, 100, Path("./test-batch"), 4)

In [ ]:
show([mask, overlaid])

In [ ]:
DS / "samples_mapillary"

In [ ]:
INITIAL_RESULTS_DIR = mkdir(NEG_MASK_WORK_DIR / "samples_mapillary" / "100")
images = list((DS / "samples_mapillary").rglob("*.jpg"))

# len(images)
run_model_on_images(images, learner100, SIZE, INITIAL_RESULTS_DIR)

In [ ]:
# generate the masks first
learner50 = load_learner(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/iter_4_engulf_t009_more-skew-resnet18-v2.pkl"
)
run_model_on_images(images, learner50, 50, mkdir(NEG_MASK_WORK_DIR / "50"))


# Generate panoptic segmasks

This would be the test output right now

In [ ]:
from mtrain.seg import mapillary as mapi, elevated_vegetation as elev
import json

SEGS = BASE / "test-pos-samples" / "segs"


def _get_cached_elev_pred(img_path):
    cand = SEGS / "elev" / f"{img_path.parent.name}.json"
    if cand.exists():
        with open(cand) as f:
            return np.array(json.load(f))
    return None


def _get_cached_mapillary_pred(img_path):
    cand = SEGS / "mapillary" / f"{img_path.parent.name}.json"
    if cand.exists():
        with open(cand) as f:
            return np.array(json.load(f))
    return None


def _get_elev_mask(img_path):
    pred = _get_cached_elev_pred(img_path)
    if pred is None:
        pred = elev.cached_predict(img_path)
    # just allow background (we dont want elevated_vegetation to be part of the output)
    neg_mask = elev.get_mask_with_labels(pred, [elev.Label.BACKGROUND])
    return neg_mask


def _get_mapillary_mask(img_path):
    pred = _get_cached_mapillary_pred(img_path)
    if pred is None:
        pred = mapi.cached_predict(img_path)
    neg_mask = mapi.get_mask_with_labels(
        pred,
        [
            mapi.Label.ROAD,
            mapi.Label.VEGETATION,
            mapi.Label.BIKE_LANE,
            mapi.Label.SIDEWALK,
            mapi.Label.SAND,
            mapi.Label.TERRAIN,
            mapi.Label.CURB,
            mapi.Label.CURB_CUT,
        ],
    )
    return neg_mask

In [ ]:
from mtrain.seg import mapillary as mapi, elevated_vegetation as elev
import json

def save_filter_masks(dirs: list[Path], dest: Path):
    for d in tqdm(dirs):
        img = d / "image.jpg"
        dest_dir = mkdir(dest / d.name)

        if not (dest_dir / "mapi.png").exists():
            mapi_pred = mapi.cached_predict(img) 
            DiskBooleanMask.save(mapi_pred.astype(np.uint8), dest_dir / "mapi.png")
        if not (dest_dir / "elev.png").exists():
            elev_pred = elev.cached_predict(img)
            DiskBooleanMask.save(elev_pred.astype(np.uint8), dest_dir / "elev.png")

In [ ]:
base_dir = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100"
)
dest = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/segs"
)
dirs = list(base_dir.glob("*"))


In [ ]:
save_filter_masks(dirs[:], dest)

## Test seg masks

In [ ]:
to_show = []
for d in dest.glob("*"):
    img_dir = base_dir / d.name
    img = DiskImage.load(img_dir / "image.jpg")
    elev_mask = DiskMaskJson.load(d / "elev.json")
    mapi_mask = DiskMaskJson.load(d / "mapi.json")
    to_show.append(img)
    to_show.append(elev_mask)
    to_show.append(mapi_mask)

In [ ]:
show(to_show, ncols=3)

# Create outputs

In [ ]:
SAMPLES_MAPILLARY = NEG_MASK_WORK_DIR / "samples_mapillary"
IMG_DIR = SAMPLES_MAPILLARY / "100"
SEGS = SAMPLES_MAPILLARY / "segs"

IMG_DIR.exists(), SEGS.exists()

In [ ]:
from mtrain.seg import mapillary as mapi, elevated_vegetation as elev

MAPI_LABELS_TO_EXCLUDE = [
    mapi.Label.PERSON,
    mapi.Label.MOTORCYCLIST,
    mapi.Label.BICYCLIST,
    mapi.Label.GROUND_ANIMAL,
    mapi.Label.OTHER_RIDER,
    mapi.Label.BIRD,

    mapi.Label.SKY,

    mapi.Label.BOAT,
    mapi.Label.BUS,
    mapi.Label.CAR,
    mapi.Label.CARAVAN,
    mapi.Label.MOTORCYCLE,
    mapi.Label.ON_RAILS,
    mapi.Label.OTHER_VEHICLE,
    mapi.Label.EGO_VEHICLE,
    mapi.Label.TRAILER,
    mapi.Label.TRUCK,
    mapi.Label.WHEELED_SLOW,
    mapi.Label.CAR_MOUNT,
    mapi.Label.BICYCLE,
    mapi.Label.BRIDGE,
    mapi.Label.TUNNEL,

    mapi.Label.BUILDING,
    mapi.Label.BILLBOARD,
    mapi.Label.BANNER,
    mapi.Label.STREET_LIGHT,
    mapi.Label.JUNCTION_BOX,
    mapi.Label.MAILBOX,
    mapi.Label.MOUNTAIN,
    mapi.Label.PHONE_BOOTH,
    mapi.Label.TRAFFIC_SIGN_FRONT,
    mapi.Label.TRAFFIC_SIGN_FRAME,
    mapi.Label.TRAFFIC_SIGN_BACK,

]

ELEV_LABELS_TO_EXCLUDE = [elev.Label.ELEVATED_VEGETATION]

def get_trimmed_mask(mask, elev_pred, mapi_pred):
    # for now, we remove all obvious things that we see
    # then we will go through all the remaining masks 
    # in decreasing order of the amount of segmentation
    # and remove more stuff
    if mapi_pred is not None:
        mapi_exclude_mask = mapi.get_mask_with_labels(mapi_pred, MAPI_LABELS_TO_EXCLUDE)
    else:
        mapi_exclude_mask = np.zeros(mask.shape, dtype=bool)
    if elev_pred is not None:
        elev_exclude_mask = elev.get_mask_with_labels(elev_pred, ELEV_LABELS_TO_EXCLUDE)
    else:
        elev_exclude_mask = np.zeros(mask.shape, dtype=bool)

    return mask & (~mapi_exclude_mask) & (~elev_exclude_mask)

def get_panoptic_masks(image_id: str):
    dest = SEGS / image_id
    if (dest / "elev.png").exists() and (dest / "mapi.png").exists():
        return DiskBooleanMask.load(dest / "elev.png"), DiskBooleanMask.load(dest / "mapi.png")
    else:
        return None, None

def get_artifacts(image_dir: Path, orig_mask_name: str = "mask.png"):
    img = DiskImage.load(image_dir / "image.jpg")
    mask = DiskBooleanMask.load(image_dir / orig_mask_name)
    elev_pred, mapi_pred = get_panoptic_masks(image_dir.name)
    out = get_trimmed_mask(mask, elev_pred, mapi_pred)
    res = DiskImage.load(image_dir / "res.jpg")

    return {
        "mask": mask,
        "img": img,
        "mapi_pred": mapi_pred,
        "elev_pred": elev_pred,
        "out_mask": out,
        "res": res,
    }

In [ ]:
dirs = (d for d in IMG_DIR.glob("*") if d.is_dir() and (d / "image.jpg").exists())
# sort these in the order shown in VSCode explorer
# for easy testing
dirs = sorted(dirs, key=lambda p: int(p.name))

In [ ]:
import cv2
import numpy as np

def extract_regions(mask: np.ndarray) -> list[dict]:
    """
    Extract connected components from a binary mask.
    Returns list of dicts with: label, area, bbox, centroid, contour
    """
    mask_u8 = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)

    regions = []
    for i in range(1, num_labels):  # skip background (0)
        x, y, w, h, area = stats[i]
        cx, cy = centroids[i]
        component_mask = (labels == i).astype(np.uint8)
        contours, _ = cv2.findContours(component_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        regions.append({
            "label": i,
            "area": int(area),
            "bbox": (int(x), int(y), int(w), int(h)),
            "centroid": (float(cx), float(cy)),
            "contour": contours[0] if contours else None,
            "component_mask": component_mask,
        })
    return regions


def region_stats(regions: list[dict]) -> dict:
    """
    Print and return stats about extracted regions.
    """
    areas = np.array([r["area"] for r in regions])
    
    percentiles = [10, 25, 50, 75, 90, 95, 99]
    pct_values = np.percentile(areas, percentiles)

    stats = {
        "num_regions": len(regions),
        "total_area": int(areas.sum()),
        "mean_area": float(areas.mean()),
        "median_area": float(np.median(areas)),
        "std_area": float(areas.std()),
        "min_area": int(areas.min()),
        "max_area": int(areas.max()),
        "percentiles": {p: float(v) for p, v in zip(percentiles, pct_values)},
    }

    print(f"Regions      : {stats['num_regions']}")
    print(f"Total area   : {stats['total_area']}")
    print(f"Mean / Std   : {stats['mean_area']:.1f} / {stats['std_area']:.1f}")
    print(f"Min / Max    : {stats['min_area']} / {stats['max_area']}")
    print("Percentiles  :")
    for p, v in stats["percentiles"].items():
        print(f"  p{p:>2}: {v:.1f}")
    return stats


def print_mask_stats(mask):
    regions = extract_regions(mask)
    region_stats(regions)

def get_mask_with_area_greater(mask, threshold):
    mask = mask.astype(bool)
    regions = extract_regions(mask)
    regions = [r for r in regions if r["area"] >= threshold]
    res = np.zeros(mask.shape, bool)
    for r in regions:
        res |= r["component_mask"].astype(bool)
    return res

In [ ]:
AREA_THRES = 10
import json

def filter_and_save_masks(meta_dir, dirs, label: str, mask_dest_name: str):
    area_and_img_id = []
    for d in tqdm(dirs):
        art = get_artifacts(d)
        res = get_mask_with_area_greater(art["out_mask"], AREA_THRES)
        area = res.sum()
        area_and_img_id.append((area, d.name))

        DiskBooleanMask.save(res, d / f"{mask_dest_name}.png")
    
    stats_f = mkdir(meta_dir) / f"{label}.json"
    stats = {
        "areas": [(int(a),i) for (a,i) in area_and_img_id],
    }
    with open(stats_f, "w") as f:
        json.dump(stats, f)
    


In [ ]:
arts = get_artifacts(dirs[0])
out = DiskBooleanMask.load(dirs[0] / "mask.png")

show([arts["img"], arts["mapi_pred"], out], ncols=3)

In [ ]:
META = mkdir(SAMPLES_MAPILLARY / "mask_finesse_pipeline")
dirs = (d for d in IMG_DIR.glob("*") if d.is_dir() and (d / "image.jpg").exists())
# sort these in the order shown in VSCode explorer
# for easy testing
dirs = sorted(dirs, key=lambda p: int(p.name))

# filter_and_save_masks(META, dirs[:], "m1", "m1")

In [ ]:
filter_and_save_masks(META, dirs, "m2", "m2")

## Walls and fence: get big intersections

In [ ]:
fence_imgs, wall_imgs = [], []

for d in tqdm(dirs):
    elev_pred, mapi_pred = get_panoptic_masks(d.name)
    mask = DiskBooleanMask.load_as_bool(d / "m1.png")

    wall_mask = mapi.get_mask_with_labels(mapi_pred, [mapi.Label.WALL]).astype(bool)
    wall_area = int((wall_mask & mask).sum())
    if wall_area > 5:
        wall_imgs.append((wall_area, d))
    

    fence_mask = mapi.get_mask_with_labels(mapi_pred, [mapi.Label.FENCE]).astype(bool)
    fence_area= int((fence_mask & mask).sum())
    if fence_area > 5:
        fence_imgs.append((fence_area, d))

In [ ]:
wall_imgs_tagged = [(a,d,"WALL") for (a, d) in reversed(sorted(wall_imgs))]
fence_imgs_tagged = [(a,d,"FENCE") for (a, d) in reversed(sorted(fence_imgs))]

len(wall_imgs_tagged), len(fence_imgs_tagged)

In [ ]:
# fence_imgs = list(reversed(sorted(fence_imgs)))
# interleaved = [x for pair in zip(wall_imgs, fence_imgs) for x in pair]
interleaved = list(itertools.chain.from_iterable(zip(wall_imgs_tagged, fence_imgs_tagged)))
interleaved[:4]

In [ ]:
import numpy as np

a = np.array([i[0] for i in interleaved])

print(f"Count:  {len(a)}")
print(f"Min:    {a.min()}")
print(f"Max:    {a.max()}")
print(f"Mean:   {a.mean():.1f}")
print(f"Median: {np.median(a):.1f}")
print(f"Std:    {a.std():.1f}")
print(f"25/75th:{np.percentile(a, 25):.1f} / {np.percentile(a, 75):.1f}")
print(f"Sum:    {a.sum()}")

In [ ]:
# claude sampling strategy, im not changing much, the graphs look okay, skewed a bit but yea
a = np.array([i[0] for i in interleaved])
n_samples = 500
n_buckets = 50  # fewer, coarser buckets

bucket_edges = np.logspace(np.log10(a.min()), np.log10(a.max() + 1), n_buckets + 1)

# figure out how many samples per bucket proportionally
buckets = []
for lo, hi in zip(bucket_edges[:-1], bucket_edges[1:]):
    idx = np.where((a >= lo) & (a < hi))[0]
    if len(idx) > 0:
        buckets.append(idx)

# sample evenly across buckets, up to 500 total
per_bucket = max(1, n_samples // len(buckets))
sampled_indices = []
for bucket in buckets:
    k = min(per_bucket, len(bucket))
    sampled_indices.extend(np.random.choice(bucket, k, replace=False))

# top up to 500 if needed
print(f"Sampled {len(sampled_indices)} indices")

# # add more
for _ in range(500 - len(sampled_indices)):
    leftover = list(set(range(500)) - set(sampled_indices))
    sampled_indices.append(random.choice(leftover))
sampled_indices = [int(i) for i in sampled_indices]

print(f"Sampled {len(sampled_indices)} indices")

In [ ]:
all_areas = np.array([i[0] for i in interleaved])

In [ ]:
plt.hist(all_areas[all_areas< 1000], 10)
plt.show()

In [ ]:
sampled_areas = np.array([interleaved[i][0] for i in sampled_indices])
plt.hist(sampled_areas[sampled_areas < 1000], 10)
plt.show()

In [ ]:
show(
    [plt.imread(interleaved[sampled_indices[0]][1] / "image.jpg"), plt.imread(interleaved[sampled_indices[1]][1] / "image.jpg")]
)

### Export Label studio format

In [ ]:
to_export = list(reversed(sorted([interleaved[i] for i in sampled_indices])))
to_export_dirs = [e[1] for e in to_export]
to_export[:2], to_export_dirs[:2]

In [ ]:
from uuid import uuid4
from pathlib import Path
import numpy as np
from label_studio_converter import brush


def split_connected_components(mask, min_area=200):
    # mask: binary (0/1)
    mask = (mask > 0).astype(np.uint8)
    num, labels = cv2.connectedComponents(mask)
    components = []
    for i in range(1, num):
        comp = (labels == i).astype(np.uint8)
        if comp.sum() >= min_area:
            components.append(comp)
    return components


# make a single annotation only right now (we do clusters later if needed)
def _make_json_part(image_path, masks, label, tmp):
    anns = []
    for mask in masks:
        dest = tmp / f"{uuid4()}.png"
        Image.fromarray(mask.astype(bool)).save(dest)
        ann = brush.image2annotation(
            dest, label, "brush", "image", model_version="v1", score=0.9
        )
        anns.append(ann["result"][0])
    path = str(Path(image_path).resolve())[1:]
    return {
        # "predictions": anns,
        "predictions": [
            {
                "model_version": "v1",
                "score": 0.88,
                "result": anns,
            }
        ],
        "data": {
            "image": f"/data/local-files/?d={path}",
        },
    }


def make_json_part(image_path, mask, label, tmp):
    masks = split_connected_components(mask)
    return _make_json_part(image_path, masks, label, tmp)

In [ ]:
import tempfile
from tqdm import tqdm

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    res = [
        make_json_part(d / "image.jpg", DiskBooleanMask.load_as_bool(d / "m1.png"), "trash", tmp) 
        for d in tqdm(to_export_dirs)
    ]

In [ ]:
with open("./m1_walls_and_fences_to_clean_anns.json", "w") as f:
    json.dump(res, f)

# Investigate data

Remove walls and fences to see how the final data approximately looks like. We will sort the final mask by area and check the big ones

In [ ]:
areas = []
for d in tqdm(dirs):
    o = get_artifacts(d, "m1.png")
    walls_and_fence = mapi.get_mask_with_labels(o["mapi_pred"], [mapi.Label.FENCE, mapi.Label.WALL])
    mask = o["mask"].astype(bool) & (~walls_and_fence)
    areas.append((mask.sum(), d))

In [ ]:
areas = list(reversed(sorted(areas)))

In [ ]:
mapi.show_seg_mask(get_artifacts(areas[idx][1], "m1.png")["mapi_pred"])

In [ ]:
idx = 130
print(f"DIR: **{areas[idx][1].resolve().relative_to(DS.parent.resolve())}**")
arts = get_artifacts(areas[idx][1], "m1.png")
show([arts["img"], arts["mask"], overlay_mask_on_img(arts["img"], arts["mask"].astype(bool))], (20,20), ncols=3)

### What is colored?

REMEMBER: This is not the final model (the current architecture picks everything, we want to remove stuff one by one). This is not good cuz there might be many things which I would miss. This is all to create a good dataset ONLY. The final model would be directly trained on that dataset

- Non road scenery is a problem (like of a shop, etc)
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/326780329730455**
- Non road scenery can include photos of a tree too technically
  - Just a closeup photo of a big tree is not recognised as a canopy by the bigger model
    - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/390833173640960**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/25060905396922455**
- Colored tree barks
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1228295912827406**
- Rocks
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/560123923759137**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1197058985243034**
- Racks near shops, which have chips etc. They also get categorized
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1126499162171737**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/765299429444398**
- I've not included POLE in the seg exclude criteria, they should be together with WALL and FENCE
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/9418880704892076**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/905336903589182**
    - This one is slightly tricky I feel
- Fallen Leaves, plant pots, Dried leaves
  - This might be particularly challenging as finding stuff within fallen leaves is hard
    - There can be huge class imbalance sometimes
  - This is basically finding garbage in vegetation, might be hard lol
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1520059529279618**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1418104656633286**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1096904281982188**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1789692645190630**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/728940703600866**
- Leaves of a shrub, this is a problem of the `elevated_vegetation` model being very pessimistic but its okay
- Lane markings
- Road which is made of stones (the border of stones is visible). Stairs of road
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/767685916162746**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1332042915174907**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1619133699086517**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1564689788273610**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1521765825265153**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1792445144693416**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/997953104075994**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1562977504666081**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1228295912827406**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/225153962280933**
- Zebra crossing and curb colored like zebra
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/176484357680452**
- Roadside shops and hawkers
  - People might even have books and all stacked on the road
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1105265378228774**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1134155588607058**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/463832161568772**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1489575072040391**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/763227394382478**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/262751362751826**
- Puddles / water on road
  - Reflection can create problems
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/508932146802554**
- Clusters are not detected
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/911967451184288**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/4256223917999636**
- Random pipes on ground are not detected
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1401981424366792**
- Big chairs should not be detected
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1265754621318173**
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/322939443724504**
- Fine gravel kinda stuff
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/25930682653190744**
- Covered cars
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/1197805371160186**
- Broken road
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/305241751213796**
- This one is quite curious: pole in front of a taxi is getting recognised as garbage
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/483415199634350**
- umbrella
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/767685916162746**
- Extreme blurs:
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/279961477168461**
- Snow
  - DIR: **datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100/457059868925393**

### Biggest winners
- Rocks vs garbage
- Hawkers and their stuff
- Road markings. Zebra crossing. Curbs
- Fallen leaves
- Find gravel
  - Just find many photos on google, run model on them, the main model cleans everything up
- Non road sceneries (like shops)
- Walls
- Fences
- Poles

### Attack strategy
- I would like to label as less data as possible
  - For stuff like walls, fences, poles, we could get data from google which has no garbage. The model will game getting negative everytime
  - It is important to keep some correct garbage samples AND make sure the model never fails them
- As an example, if i'm doing leaves thing 
  - Keep examples of leaves only
    - All empty, model can game it
  - Keep examples of no leaves
    - This would simply be the input mask as the output, model can game it too
  - Keep examples of mixed (these would be less)
    - This would less data, this data should have more weight attached?
  - Leaves is specifically difficult


#### lane markings
- For lane markings etc, I can simply find the ones with lane markings and not keep them in data. 
- Keep the ones with lane markings, fix the lane markings. 
- From google, we get data without garbage and with lane markings, we can automate labelling it (since there is no garbage in the road)
- Run model on this data. Create clean data with this model. Continue to next model.  
- This should be good for lane markings I think

#### rocks
- This too, same as lane markings I feel. Although I don't have a segmentor for rocks shiz
  - I could make a model which classifies them though, given a mask. It would find the crop and classify whether that region has a rock.  
  - Should be easy
  - I think this is the easiest one right now. I'll start with rocks


- Wait, I'm creating a top level classifier on this vs that right?
  - Leaf vs other stuff is also fine I think
  - Can I just use this? We would have extremely small masks though, would we be able to find without context? I dont think so
  - If I just use a big photo, then it would just game it, and mark non-leaves leaves too

Goal for leaves:
  - Model marks dry leaves, leaves, etc. as garbage
  - Trash can be strewn mid leaves
  - The simplest way is to take a pic around the surrounding region and classify whether it is leaf or not. In this case, I need the average mask size, then I can take leaf photos, resize them and run my model against it. 
    - I will need a generic dataset of "objects" which I keep everywhere (maybe my TACO dataset is fine)

# Test outputs

Just checking how stuff is looking. Although simple segmentation masks are not very reliable I feel, I might need to train the whole model without the masks. for now, im just using the segmask outputs as output labels for my dataset work. Initial model harness testing.  

In [ ]:
bd = NEG_MASK_WORK_DIR / "50" / "133855545702381"

img = DiskImage.load(bd / "image.jpg")
orig_mask = DiskBooleanMask.load_as_bool(bd / "mask.png")
out_mask = DiskBooleanMask.load_as_bool(bd / "out_mask.png")
seg_filter_mask = DiskBooleanMask.load_as_bool(bd / "seg_filter_mask.png")

show([img, orig_mask, out_mask, seg_filter_mask])

In [ ]:
show(
    [
        overlay_mask_on_img(img, out_mask),
        out_mask,
        overlay_mask_on_img(img, orig_mask),
        orig_mask,
    ]
)

In [ ]:
plt.imshow(_get_mapillary_mask(bd / "image.jpg"))

In [ ]:
pred = _get_cached_mapillary_pred(bd / "image.jpg")

In [ ]:
_, ax = plt.subplots(1, 1)
ax.imshow(img)
draw_grid(ax, img.shape, 224)

In [ ]:
np.unique(pred[400:425, 125:150])

# Tiling
We have very bad resolution, we simply get tiles from the images instead of resizing. We dont want to resize at all.   
Btw, it makes sense to train on sizes of factor 32, apparently resnet is good with them. I will need to do that once in my models also.  

In [ ]:
def tile_image(img, N, stride):
    """
    img: numpy array (H, W, C)
    N: tile size
    stride: step size between tiles
    Returns list of (tile, (row_start, col_start)) tuples
    """
    H, W = img.shape[:2]
    tiles = []

    row_starts = list(range(0, H - N, stride)) + [H - N]
    col_starts = list(range(0, W - N, stride)) + [W - N]

    # deduplicate while preserving order
    row_starts = sorted(set(row_starts))
    col_starts = sorted(set(col_starts))

    for r in row_starts:
        for c in col_starts:
            tile = img[r : r + N, c : c + N]
            tiles.append(tile)

    return tiles

In [ ]:
# test
bd = NEG_MASK_WORK_DIR / "50" / "133855545702381"

img = DiskImage.load(bd / "image.jpg")
orig_mask = DiskBooleanMask.load_as_bool(bd / "mask.png")
out_mask = DiskBooleanMask.load_as_bool(bd / "out_mask.png")
seg_filter_mask = DiskBooleanMask.load_as_bool(bd / "seg_filter_mask.png")


img_tiles = tile_image(img, 224, 112)
mask_tiles = tile_image(out_mask, 224, 112)


show(img_tiles)

In [ ]:
show(mask_tiles)

# Tiled dataset

- The first thing we do is use the existing dataset
- Tile each image and mask -> gives up a good number of data points
- Keep 3x empty tiles compared to tiles with non zero mask

In [ ]:
def get_filtered_tiles(in_d: Path, N: int, stride: int):
    img = DiskImage.load(in_d / "image.jpg")
    out_mask = DiskBooleanMask.load(in_d / "out_mask.png")
    mask = DiskBooleanMask.load(in_d / "mask.png")

    image_tiles = tile_image(img, N, stride)
    out_mask_tiles = tile_image(out_mask, N, stride)
    mask_tiles = tile_image(mask, N, stride)

    it = zip(image_tiles, mask_tiles, out_mask_tiles)
    pos_samples, empty_samples = [], []
    for imt, mt, omt in it:
        if omt.sum() + mt.sum() > 0:
            pos_samples.append((imt, mt, omt))
        else:
            empty_samples.append((imt, mt, omt))

    random.shuffle(pos_samples)
    random.shuffle(empty_samples)
    empty_samples = empty_samples[: 3 * len(pos_samples)]

    res = pos_samples + empty_samples
    random.shuffle(res)
    return res

In [ ]:
bd = NEG_MASK_WORK_DIR / "50" / "133855545702381"

tiles = get_filtered_tiles(bd, 224, 112)
show(tiles[0], ncols=3)

In [ ]:
def make_tiled_dataset(orig_root: Path, dest: Path, N: int, stride: int):
    dest = mkdir(dest)
    ds = list(orig_root.glob("*"))
    for d in tqdm(ds):
        if not d.is_dir():
            continue
        tiles = get_filtered_tiles(d, N, stride)
        for i, tile in enumerate(tiles):
            dest_dir = mkdir(dest / f"{d.name}_{i}")
            image, mask, out_mask = tile
            DiskImage.save(image, dest_dir / "image.jpg")
            DiskBooleanMask.save(mask, dest_dir / "mask.png")
            DiskBooleanMask.save(out_mask, dest_dir / "out_mask.png")

In [ ]:
tiled_dir = mkdir(NEG_MASKING_DIR / "tiled")
make_tiled_dataset(NEG_MASK_WORK_DIR / "100", tiled_dir, 224, 112)

## Testing

In [ ]:
sample_dir = tiled_dir / "133855545702381_37"

img = DiskImage.load(sample_dir / "image.jpg")
mask = DiskBooleanMask.load_as_bool(sample_dir / "mask.png")
out_mask = DiskBooleanMask.load_as_bool(sample_dir / "out_mask.png")

show([img, mask, out_mask], ncols=3)

# Model harness: datasets and dataloaders

Following: https://docs.fast.ai/tutorial.siamese.html

In [ ]:
ROOT_DATA_DIR = NEG_MASKING_DIR / "tiled"

In [ ]:
def _is_valid_dir(direc: Path):
    valid = (
        (direc / "image.jpg").exists()
        and (direc / "mask.png").exists()
        and (direc / "out_mask.png").exists()
    )
    if not valid:
        print(
            f"WARN: directory: {direc} is not valid, it does not contain all required files"
        )
    return valid


class MaskFixDataset(torch.utils.data.Dataset):
    def __init__(
        self, dirs: list[Path | str], norm_stats: tuple[list[float], list[float]]
    ):
        self.mean = torch.tensor(norm_stats[0])
        self.std = torch.tensor(norm_stats[1])
        self.dirs = [d for d in map(Path, dirs) if _is_valid_dir(d)]

    def __len__(self):
        return len(self.dirs)

    # def __getit
    def __getitem__(self, index):
        d = self.dirs[index]
        img = torch.Tensor(DiskImage.load(d / "image.jpg"))
        img = img.permute(2, 0, 1).float() / 255.0
        img = (img - self.mean[:, None, None]) / self.std[:, None, None]
        mask = torch.Tensor(DiskBooleanMask.load(d / "mask.png"))
        mask = mask.float().unsqueeze(0)
        combined = torch.cat([img, mask], dim=0)
        out_mask = torch.Tensor(DiskBooleanMask.load(d / "out_mask.png"))

        # flipping, not doing rn, not very necessary for now
        # if random.random() > 0.5:
        #     combined = torch.flip(combined, dims=[-1])
        #     out_mask = torch.flip(out_mask, dims=[-1])

        return (combined, out_mask)

In [ ]:
from fastai.vision.all import imagenet_stats

dirs = list(ROOT_DATA_DIR.glob("*"))
idxs = np.random.permutation(range(len(dirs)))
cut = int(0.8 * len(dirs))

train_dirs = [dirs[idx] for idx in idxs[:cut]]
valid_dirs = [dirs[idx] for idx in idxs[cut:]]

train_ds = MaskFixDataset(train_dirs, imagenet_stats)
valid_ds = MaskFixDataset(valid_dirs, imagenet_stats)

In [ ]:
from fastai.data.core import DataLoaders

dls = DataLoaders.from_dsets(train_ds, valid_ds)

In [ ]:
from fastai.vision.all import (
    unet_learner,
    resnet18,
    BCEWithLogitsLossFlat,
    ProgressCallback,
    store_attr,
    DiceLoss,
)


class CombinedLoss:
    def __init__(self, axis=1, smooth=1.0, alpha=1.0):
        store_attr()
        self.bce_loss = BCEWithLogitsLossFlat(axis=axis)
        self.dice_loss = DiceLoss(axis, smooth)

    def __call__(self, pred, targ):
        return self.bce_loss(pred, targ) + self.alpha * self.dice_loss(pred, targ)

    def decodes(self, x):
        return (x.sigmoid() > 0.5).long()

    def activation(self, x):
        return x.sigmoid()


learn = unet_learner(
    dls,
    resnet18,
    n_in=4,
    n_out=1,
    normalize=False,
    pretrained=True,
    loss_func=BCEWithLogitsLossFlat(),
)
learn.remove_cb(ProgressCallback)

In [ ]:
learn.fine_tune(1)

In [ ]:
b = dls.one_batch()

# Merge 50 and 100 masks

In [ ]:
b[0].shape

In [ ]:
dir50 = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/50"
)
dir100 = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100"
)
dest = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/merge_50_100"
)

In [ ]:
dirs_to_merge = list(dir100.glob("*"))

In [ ]:
import shutil

for d in tqdm(dirs_to_merge):
    dest_dir = mkdir(dest / d.name)
    shutil.copy(d / "image.jpg", dest_dir)
    mask = DiskBooleanMask.load_as_bool(d / "mask.png")
    if (dir50 / d.name / "mask.png").exists():
        mask50 = DiskBooleanMask.load_as_bool(dir50 / d.name / "mask.png")
        mask = mask50 | mask
    res = overlay_mask_on_img(DiskImage.load(d / "image.jpg"), mask)
    DiskImage.save(res, dest_dir / "res.jpg")
    DiskBooleanMask.save(mask.astype(np.uint8), dest_dir / "mask.png")

## Test merge

In [ ]:
test_dir_name = dirs_to_merge[19].name

test_d_100 = dir100 / test_dir_name
test_d_50 = dir50 / test_dir_name
test_d_merge = dest / test_dir_name


test_d_100.exists(), test_d_50.exists(), test_d_merge.exists()

In [ ]:
to_show = []
cands = [test_d_100, test_d_50, test_d_merge]
for cand in cands:
    to_show.append(DiskImage.load(cand / "image.jpg"))
for cand in cands:
    to_show.append(DiskBooleanMask.load_as_bool(cand / "mask.png"))
for cand in cands:
    to_show.append(DiskImage.load(cand / "res.jpg"))

show(
    to_show,
    ncols=3,
)